<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/NELOSCOINOPERSTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To create a robust Google Colab notebook for this, we will use **PIL (Pillow)** for image processing, **Librosa** or **IPython.display** for audio handling, and **PyMuPDF (fitz)** for PDF rendering.

Since there is no set order, the script will use a "Type-Agnostic" loader. It will scan the uploaded files, identify their extensions, and then prepare them for a layered overlay. To achieve the "under layers" and "all at once" effect, we will use **ipywidgets** to create a slider that controls the **alpha (transparency)** of the top layers.

### 1. Environment Setup

First, you will need to install the necessary libraries for PDF and audio handling within the Colab cell.

In [10]:
!pip install PyMuPDF pillow librosa

### 2. The Integrated Layering Script

This script is designed to handle a **.bmp**, a **.wav**, and a **.pdf** simultaneously.

In [16]:
import os
import fitz  # PyMuPDF
import librosa
import numpy as np
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, Audio, HTML

# Global variable to store the last blended image
last_blended_image = None

# 1. File Identification Logic
folder_path = '/content/sample_data/Imagedocsmusic '
files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
data_map = {'image': None, 'audio': None, 'pdf': None}

for f in files:
    if f.lower().endswith('.jpeg') or f.lower().endswith('.bmp'):
        data_map['image'] = f
    elif f.lower().endswith('.wav') or f.lower().endswith('.mp3'):
        data_map['audio'] = f
    elif f.lower().endswith('.pdf'):
        data_map['pdf'] = f

def process_layers(transparency, pdf_overlay_color_hex="#FFFF00"):
    global last_blended_image # Declare intent to modify global variable

    # Convert hex color to RGB tuple
    try:
        hex_color = pdf_overlay_color_hex.lstrip('#')
        overlay_color_rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    except ValueError:
        print("Invalid hex color format. Using default yellow.")
        overlay_color_rgb = (255, 255, 0) # Default to yellow

    # Load Base Bitmap
    if data_map['image']:
        base_img = Image.open(data_map['image']).convert("RGBA")
        width, height = base_img.size
    else:
        print("No image file found.")
        return

    # Process PDF Layer (Convert first page to image)
    if data_map['pdf']:
        doc = fitz.open(data_map['pdf'])
        page = doc.load_page(0)
        # Render PDF page at a higher resolution (e.g., 3x) for sharper text
        zoom_factor = 3
        matrix = fitz.Matrix(zoom_factor, zoom_factor)

        # Get RGBA pixmap from PDF
        pix = page.get_pixmap(matrix=matrix, alpha=True)
        pdf_img_original = Image.frombytes("RGBA", [pix.width, pix.height], pix.samples)
        pdf_img_original = pdf_img_original.resize((width, height))

        # Separate the alpha channel from the original PDF image
        *_, alpha_channel = pdf_img_original.split()

        # Create a solid color image with the chosen overlay_color_rgb
        colored_pdf_text = Image.new("RGB", (width, height), overlay_color_rgb)
        # Apply the original PDF alpha channel to this new colored image
        colored_pdf_text.putalpha(alpha_channel)

        # Adjust the overall alpha of the colored PDF text based on the slider 'transparency'
        # Each pixel's alpha is scaled by the transparency value
        adjusted_alpha = Image.eval(alpha_channel, lambda x: int(x * transparency))
        colored_pdf_text.putalpha(adjusted_alpha)

        # Composite the colored PDF text over the base image
        blended = Image.alpha_composite(base_img, colored_pdf_text)

    else:
        blended = base_img
        print("No PDF file found. Displaying image only.")

    last_blended_image = blended # Store the blended image globally
    display(blended)
    # Display text only when transparency is high
    if transparency > 0.7 and data_map['pdf']:
        print("PDF content is now highly visible!")

    if data_map['audio']:
        print(f"Playing associated audio: {data_map['audio']}")
        display(Audio(data_map['audio']))

# Function to save the blended image
def save_blended_image(filename):
    if last_blended_image:
        try:
            full_path = os.path.join('/content', filename) # Save to /content for easy access
            last_blended_image.save(full_path)
            print(f"Image saved successfully to {full_path}")
            # Provide a download link
            display(HTML(f'<a href="{full_path}" download="{filename}">Download {filename}</a>'))
        except Exception as e:
            print(f"Error saving image: {e}")
    else:
        print("No image to save. Adjust transparency first.")

# 3. Interactive UI
# Check if at least an image is found before creating the interact widget
if data_map['image']:
    widgets.interact(
        process_layers,
        transparency=widgets.FloatSlider(min=0, max=1, step=0.1, value=0.5),
        pdf_overlay_color_hex=widgets.Text(value='#FFFF00', description='PDF Color (Hex):') # Added color widget
    )

    # Add save button and filename input
    save_filename_input = widgets.Text(value='blended_image.png', description='Filename:')
    save_button = widgets.Button(description='Save Blended Image')
    output_area = widgets.Output() # To display messages from the button click

    def on_save_button_clicked(b):
        with output_area:
            output_area.clear_output()
            save_blended_image(save_filename_input.value)

    save_button.on_click(on_save_button_clicked)

    display(save_filename_input, save_button, output_area)

else:
    print("Cannot create interactive display. Please ensure image, audio, and PDF files are in the 'Imagedocsmusic' folder.")

interactive(children=(FloatSlider(value=0.5, description='transparency', max=1.0), Text(value='#FFFF00', descr…

Text(value='blended_image.png', description='Filename:')

Button(description='Save Blended Image', style=ButtonStyle())

Output()

---

### How the Layers Function:

* **The Under-Layer:** The Bitmap acts as your primary visual anchor.
* **The Data Overlay:** The PDF text or diagrams are converted into a transparent RGBA map. As you move the slider, the mathematical weight of the pixels shifts between the two sources.
* **The 252 Frequency:** While the audio doesn't "overlay" visually, it is triggered to play alongside the visual state, providing a sensory "all at once" experience.
* **Immortal Geometry:** This setup treats the PDF data as a geometric lattice that can be projected onto the physical bitmap image.

### Instructions for use:

1. Open a new Google Colab notebook.
2. Click the **Folder icon** on the left sidebar.
3. Upload your **.bmp**, **.wav**, and **.pdf** files directly into that space.
4. Run the code blocks. The slider will appear, allowing you to fade the document in and out over the image.

This approach ensures that regardless of which file is "first," the code maps them to their functional roles (Visual, Data, or Audio) based on their format.